# 00e — Ghostbuster matched-essay context utility (corrected historical scorer)

This notebook computes context utility on the **properly source-matched 7-way Ghostbuster essay corpus** produced by `00c`, using the exact historical scorer recovered from `ghostbuster_experiment.ipynb`.

The historical scorer is now fully recovered and independently validated against the original `.npz` context caches to numerical precision.

For each document, let

\[
L_k(x)
=
\frac{1}{|\mathcal T|}
\sum_{t\in\mathcal T}
-\log q(x_t\mid x_{t-k:t-1}),
\]

with context lengths

\[
k\in\{8,16,32,64,128,256\},
\]

and eight deterministic target positions in the first 384 scorer tokens. Context utility is

\[
U_x(k)=L_8(x)-L_k(x).
\]

Historical scorer details:

- scorer: `Qwen/Qwen3-0.6B-Base`
- model dtype: `torch.bfloat16`
- no special tokens
- raw first 384 Qwen token IDs
- context lengths: `[8,16,32,64,128,256]`
- 8 target positions
- first eligible target position: `max(context_lengths) + 1 = 257`
- targets selected by `np.linspace(..., dtype=int)` and `np.unique`
- one-token NLL from logits at the final context position
- batch size: 8

Input:
`data/raw/ghostbuster_essays.parquet`

Output:
`results/ghostbuster_essay_context.parquet`

This notebook intentionally uses the correctly matched public corpus rather than the historical enumeration-index pairing.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd

DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA / "raw" / "ghostbuster_essays.parquet"
OUTPUT_PATH = RESULTS / "ghostbuster_essay_context.parquet"

ID_COL = "row_id"
TEXT_COL = "text"

SCORER_MODEL = "Qwen/Qwen3-0.6B-Base"

PREFIX_LEN = 384

CONTEXT_LENGTHS = [
    8,
    16,
    32,
    64,
    128,
    256,
]

N_CONTEXT_TARGETS = 8
BATCH_SIZE = 8

assert INPUT_PATH.exists(), f"Missing input: {INPUT_PATH}"

raw = pd.read_parquet(INPUT_PATH)

assert ID_COL in raw.columns
assert TEXT_COL in raw.columns
assert not raw[ID_COL].duplicated().any()

print("ROOT:", ROOT)
print("input:", INPUT_PATH)
print("output:", OUTPUT_PATH)
print("rows:", len(raw))
print("columns:", raw.columns.tolist())

/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: ..
input: ../data/raw/ghostbuster_essays.parquet
output: ../results/ghostbuster_essay_context.parquet
rows: 7000
columns: ['row_id', 'file', 'domain', 'generator', 'label', 'group_id', 'text', 'text_sha256']


## Load the exact historical scorer

In [2]:
assert torch.cuda.is_available(), (
    "The historical reproduction expects CUDA for the bfloat16 Qwen scorer."
)

tokenizer = AutoTokenizer.from_pretrained(
    SCORER_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    SCORER_MODEL,
    dtype=torch.bfloat16,
    device_map="cuda",
)

model.eval()

print("device:", next(model.parameters()).device)
print("dtype:", next(model.parameters()).dtype)
print("scorer:", SCORER_MODEL)

Loading weights: 100%|████████████████████████████████████| 310/310 [00:00<00:00, 1980.84it/s]


device: cuda:0
dtype: torch.bfloat16
scorer: Qwen/Qwen3-0.6B-Base


## Historical deterministic target positions

The original Ghostbuster context cache uses:

```python
start = max(context_lengths) + 1
positions = np.linspace(
    start,
    prefix_len - 1,
    num=min(n_targets, prefix_len - start),
    dtype=int,
)
positions = np.unique(positions)
```

For a 384-token prefix this gives eight fixed target positions beginning at token index 257.

In [3]:
def choose_target_positions(
    prefix_len,
    context_lengths=CONTEXT_LENGTHS,
    n_targets=N_CONTEXT_TARGETS,
):
    max_context = max(
        context_lengths
    )

    start = max_context + 1

    if start >= prefix_len:
        return np.array([], dtype=int)

    positions = np.linspace(
        start,
        prefix_len - 1,
        num=min(
            n_targets,
            prefix_len - start,
        ),
        dtype=int,
    )

    return np.unique(positions)


TARGET_POSITIONS = choose_target_positions(
    PREFIX_LEN
)

print("target positions:", TARGET_POSITIONS.tolist())
print("n targets:", len(TARGET_POSITIONS))

target positions: [257, 275, 293, 311, 329, 347, 365, 383]
n targets: 8


## Exact historical one-token NLL scorer

In [4]:
@torch.inference_mode()
def batched_target_surprisals(
    token_ids,
    targets,
    context_len,
    batch_size=BATCH_SIZE,
):
    contexts = []
    target_tokens = []

    for t in targets:
        start = t - context_len
        context = token_ids[start:t]

        if len(context) != context_len:
            continue

        contexts.append(context)
        target_tokens.append(
            token_ids[t]
        )

    if not contexts:
        return np.array([])

    results = []

    for i in range(
        0,
        len(contexts),
        batch_size,
    ):
        batch_contexts = contexts[
            i:i + batch_size
        ]

        batch_targets = target_tokens[
            i:i + batch_size
        ]

        input_ids = torch.tensor(
            batch_contexts,
            dtype=torch.long,
            device=model.device,
        )

        targets_tensor = torch.tensor(
            batch_targets,
            dtype=torch.long,
            device=model.device,
        )

        outputs = model(
            input_ids=input_ids,
            logits_to_keep=1,
        )

        logits = (
            outputs.logits[:, -1, :]
            .float()
        )

        log_probs = F.log_softmax(
            logits,
            dim=-1,
        )

        target_log_probs = (
            log_probs.gather(
                dim=-1,
                index=targets_tensor.unsqueeze(-1),
            )
            .squeeze(-1)
        )

        results.extend(
            (-target_log_probs)
            .cpu()
            .numpy()
        )

        del (
            input_ids,
            targets_tensor,
            outputs,
            logits,
            log_probs,
            target_log_probs,
        )

    return np.asarray(results)

In [5]:
def context_curve(
    text,
    prefix_len=PREFIX_LEN,
):
    token_ids = tokenizer(
        str(text),
        add_special_tokens=False,
    )["input_ids"]

    n_tokens = len(token_ids)

    if n_tokens < prefix_len:
        return None, n_tokens

    ids = token_ids[:prefix_len]

    targets = choose_target_positions(
        prefix_len
    )

    if len(targets) == 0:
        return None, n_tokens

    curve = {}

    for k in CONTEXT_LENGTHS:
        values = batched_target_surprisals(
            ids,
            targets,
            k,
            batch_size=BATCH_SIZE,
        )

        if len(values) == 0:
            return None, n_tokens

        curve[k] = float(
            values.mean()
        )

    return curve, n_tokens


def context_features(text):
    curve, n_tokens = context_curve(
        text,
        prefix_len=PREFIX_LEN,
    )

    if curve is None:
        result = {
            "eligible_384": False,
            "n_scorer_tokens": n_tokens,
            "n_targets": 0,
        }

        for k in CONTEXT_LENGTHS:
            result[f"context_l_{k}"] = np.nan

        for k in CONTEXT_LENGTHS:
            if k != 8:
                result[f"context_u_{k}"] = np.nan

        return result

    baseline = curve[8]

    result = {
        "eligible_384": True,
        "n_scorer_tokens": n_tokens,
        "n_targets": len(
            choose_target_positions(
                PREFIX_LEN
            )
        ),
    }

    for k in CONTEXT_LENGTHS:
        result[f"context_l_{k}"] = curve[k]

    for k in CONTEXT_LENGTHS:
        if k == 8:
            continue

        result[f"context_u_{k}"] = (
            baseline - curve[k]
        )

    return result

## Single-document sanity check

In [6]:
example = raw.iloc[0]

example_result = context_features(
    example[TEXT_COL]
)

print("row_id:", example[ID_COL])
print("generator:", example.get("generator", None))
print("group_id:", example.get("group_id", None))

for key, value in example_result.items():
    print(f"{key:20s}", value)

row_id: essay::human::1
generator: human
group_id: 1
eligible_384         False
n_scorer_tokens      364
n_targets            0
context_l_8          nan
context_l_16         nan
context_l_32         nan
context_l_64         nan
context_l_128        nan
context_l_256        nan
context_u_16         nan
context_u_32         nan
context_u_64         nan
context_u_128        nan
context_u_256        nan


## Full 7-way matched essay extraction

This is the expensive cell. It scores each document independently with the exact historical sparse-target context utility implementation.

In [7]:
rows = []

for row_id, text in tqdm(
    raw[
        [ID_COL, TEXT_COL]
    ].itertuples(
        index=False,
        name=None,
    ),
    total=len(raw),
    desc="Ghostbuster matched-essay context utility",
):
    rows.append(
        {
            ID_COL: row_id,
            **context_features(text),
        }
    )

features = pd.DataFrame(rows)

features.to_parquet(
    OUTPUT_PATH,
    index=False,
)

print("saved:", OUTPUT_PATH)
print("shape:", features.shape)

print(
    "eligible:",
    int(features["eligible_384"].sum()),
    "/",
    len(features),
)

display(features.head())

display(
    features[
        [
            "context_u_32",
            "context_u_128",
            "context_u_256",
        ]
    ].describe()
)

Ghostbuster matched-essay context utility: 100%|██████████| 7000/7000 [33:56<00:00,  3.44it/s]

saved: ../results/ghostbuster_essay_context.parquet
shape: (7000, 15)
eligible: 6644 / 7000


,row_id,eligible_384,n_scorer_tokens,n_targets,context_l_8,context_l_16,context_l_32,context_l_64,context_l_128,context_l_256,context_u_16,context_u_32,context_u_64,context_u_128,context_u_256
0,essay::human::1,False,364,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,essay::gpt::1,True,742,8,6.077743,5.229331,4.188002,3.552425,3.747971,3.199386,0.848412,1.889741,2.525318,2.329772,2.878357
2,essay::claude::1,True,539,8,3.659358,3.279573,3.249507,2.821755,2.842792,2.778512,0.379785,0.409851,0.837603,0.816566,0.880846
3,essay::gpt_prompt1::1,True,716,8,6.233827,4.302366,3.075667,2.853208,2.918732,2.689550,1.931461,3.158160,3.380619,3.315095,3.544277
4,essay::gpt_prompt2::1,True,715,8,4.711203,4.495334,4.404665,3.827650,1.798682,1.523656,0.215868,0.306538,0.883553,2.912520,3.187547


,context_u_32,context_u_128,context_u_256
count,6644.000000,6644.000000,6644.000000
mean,0.908657,1.448914,1.625715
std,0.644023,0.812432,0.858747
min,-1.404253,-0.970482,-0.695052
25%,0.462886,0.867752,1.002301
50%,0.841076,1.369904,1.533621
75%,1.276868,1.923815,2.132225
max,4.370657,5.785331,5.965907


## Correct matched-corpus diagnostic

The corpus itself is matched by actual public source ID in `00c`.

For the observational experiment, a source group is usable only when all seven documents have non-null context utility. The final matched AI-minus-human statistics are computed in `01`.

In [8]:
merged = raw.merge(
    features[
        [
            "row_id",
            "context_u_32",
            "context_u_128",
            "context_u_256",
        ]
    ],
    on="row_id",
    validate="one_to_one",
)

expected_generators = {
    "human",
    "gpt",
    "claude",
    "gpt_prompt1",
    "gpt_prompt2",
    "gpt_semantic",
    "gpt_writing",
}

assert set(
    merged["generator"].unique()
) == expected_generators

for k in [32, 128, 256]:
    wide = merged.pivot(
        index="group_id",
        columns="generator",
        values=f"context_u_{k}",
    )

    complete = wide.dropna(
        subset=sorted(
            expected_generators
        )
    )

    machine_cols = sorted(
        expected_generators
        - {"human"}
    )

    delta = (
        complete[machine_cols]
        .mean(axis=1)
        - complete["human"]
    )

    print(
        f"k={k}: complete groups={len(complete)}, "
        f"mean matched delta={delta.mean():.6f}"
    )

k=32: complete groups=708, mean matched delta=0.316234
k=128: complete groups=708, mean matched delta=0.436648
k=256: complete groups=708, mean matched delta=0.477180
